In [1]:
%pip install rapidfuzz -q

import re
import pandas as pd
import numpy as np

from rapidfuzz import fuzz

Note: you may need to restart the kernel to use updated packages.


In [2]:
df = pd.read_csv('../data/processed/ecommerce_product_raw.csv')
df

,nama_produk,harga,rating,jumlah_terjual,lokasi_penjual,brand,kategori,kondisi,platform
0,motorolla MOTO G45 5G RAM 8+8/256GB Garansi Re...,Rp1.500.000,NaN,250+ barang berhasil terjual,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia
1,ZTE Nubia Z50 (5G) (12GB/256GB) (The Best Of C...,Rp9.499.000,NaN,NaN,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia
2,ZTE Nubia Redmagic 8s Pro 16GB/512GB Garansi R...,Rp14.999.000,NaN,NaN,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia
3,ZTE Nubia Z50 (5G) (12GB/256GB) (Jaminan Produ...,Rp9.499.000,NaN,NaN,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia
4,Digitalisme x Promo Nubia RedMagic 11 Pro 16GB...,Rp17.652.000,5.0,Terjual 6,Kota Administrasi Jakarta Pusat,Zte,Hp,Baru,Tokopedia
...,...,...,...,...,...,...,...,...,...
97161,TAM | Asus Zenfone 9 & 8 16/256GB 8/128GB 256 ...,Rp3.775.000,4.8,Terjual 7,Kota Tangerang Selatan,Asus,Hp,Bekas,Tokopedia
97162,ASUS ROG PHONE 3 5G 12GB/256GB 8GB/128GB GARAN...,Rp3.555.000,4.7,Terjual 100+,Kota Tangerang,Asus,Hp,Bekas,Tokopedia
97163,Asus Rog Phone 3 Original 128GB Termurah Berga...,Rp3.509.999,NaN,NaN,Kab. Bogor,Asus,Hp,Bekas,Tokopedia
97164,ASUS ZENFONE 9 5G 8/256GB Produk Resmi Indonesia,Rp5.999.000,5.0,Terjual 6,Kota Administrasi Jakarta Selatan,Asus,Hp,Bekas,Tokopedia


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 97166 entries, 0 to 97165
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nama_produk     96880 non-null  str  
 1   harga           96215 non-null  str  
 2   rating          28424 non-null  str  
 3   jumlah_terjual  46853 non-null  str  
 4   lokasi_penjual  44889 non-null  str  
 5   brand           97166 non-null  str  
 6   kategori        97166 non-null  str  
 7   kondisi         75646 non-null  str  
 8   platform        97166 non-null  str  
dtypes: str(9)
memory usage: 15.8 MB


In [4]:
df.describe()

,nama_produk,harga,rating,jumlah_terjual,lokasi_penjual,brand,kategori,kondisi,platform
count,96880,96215,28424,46853,44889,97166,97166,75646,97166
unique,28060,9245,45,921,287,27,2,2,5
top,Akan hadir,1.999.000,5.0,Terjual 1,Kota Surabaya,Oppo,Hp,Baru,Blibli
freq,645,955,10570,8285,4512,16290,67001,62680,48778


In [5]:
mask_akan_hadir = df['nama_produk'].str.strip().str.lower() == 'akan hadir'
print(f"Drop 'Akan Hadir': {mask_akan_hadir.sum()} baris")
df = df[~mask_akan_hadir]

Drop 'Akan Hadir': 645 baris


In [6]:
print(f"Duplikat ditemukan: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape setelah drop duplikat: {df.shape}")

Duplikat ditemukan: 60437
Shape setelah drop duplikat: (36084, 9)


In [7]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [8]:
df.isnull().sum()

nama_produk          38
harga                42
rating            18969
jumlah_terjual    14204
lokasi_penjual    10222
brand                 0
kategori              0
kondisi            2705
platform              0
dtype: int64

In [9]:
df = df.dropna(subset=['nama_produk', 'harga'])
print(f"Shape setelah drop baris kritis: {df.shape}")

Shape setelah drop baris kritis: (36041, 9)


In [10]:
df['harga'] = df['harga'].astype(str)
df['harga'] = df['harga'].str.replace('Rp', '', regex=False)
df['harga'] = df['harga'].str.replace(r'\D', '', regex=True)
df['harga'] = pd.to_numeric(df['harga'], errors='coerce')

# Filter harga tidak wajar (< 100rb tidak masuk akal untuk HP/Laptop)
df = df[df['harga'] >= 100_000]
print(f"Shape setelah filter harga tidak wajar: {df.shape}")
print("Statistik harga:")
print(df['harga'].describe())

Shape setelah filter harga tidak wajar: (35651, 9)
Statistik harga:
count    3.565100e+04
mean     5.784469e+06
std      7.342953e+06
min      1.000000e+05
25%      1.800000e+06
50%      3.499000e+06
75%      7.149000e+06
max      1.990000e+08
Name: harga, dtype: float64


In [11]:
df['rating'] = df['rating'].astype(str).str.replace(',', '.', regex=False)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating'] = df['rating'].where(df['rating'].between(1, 5), np.nan)

In [12]:
def parse_jumlah_terjual(j):
    if pd.isna(j):
        return np.nan
    j = str(j).replace('\xa0', ' ').strip()
    # Tangkap pola "rb+" → ribuan
    rb = re.search(r'(\d+(?:[.,]\d+)?)\s*rb\+?', j, re.IGNORECASE)
    if rb:
        return int(float(rb.group(1).replace(',', '.')) * 1000)
    # Tangkap angka biasa
    num = re.search(r'(\d+)', j)
    return int(num.group(1)) if num else np.nan

df['jumlah_terjual'] = df['jumlah_terjual'].apply(parse_jumlah_terjual)
df['jumlah_terjual'] = df['jumlah_terjual'].fillna(0).astype(int)

In [13]:
df['brand'].unique()

<ArrowStringArray>
[       'Zte', 'Blackshark',     'Lenovo',       'Sony',      'Nokia',
      'Honor',     'Huawei',       'Asus',     'Iphone',       'Itel',
      'Tecno',    'Infinix',     'Realme',       'Vivo',       'Oppo',
       'Poco',     'Xiaomi',    'Samsung',  'Microsoft',       'Acer',
         'Hp',      'Zyrex',      'Advan',      'Axioo',       'Dell',
        'Msi',      'Apple']
Length: 27, dtype: str

In [14]:
df['brand'] = df['brand'].replace({'Iphone': 'Apple', 'iphone': 'Apple'})

# Override brand berdasarkan deteksi keyword di nama_produk
iphone_mask = df['nama_produk'].str.contains(
    r'\biphone\b|\bapple\s+iphone\b', case=False, na=False, regex=True
)
df.loc[iphone_mask, 'brand'] = 'Apple'

print("Brand unik setelah fix:")
print(df['brand'].value_counts())

Brand unik setelah fix:
brand
Oppo          2652
Vivo          2245
Realme        2093
Infinix       1968
Apple         1862
Lenovo        1862
Samsung       1847
Asus          1834
Xiaomi        1736
Hp            1455
Nokia         1454
Huawei        1414
Acer          1410
Advan         1190
Axioo         1184
Msi           1147
Dell          1048
Tecno          931
Itel           903
Microsoft      898
Honor          893
Poco           866
Zyrex          796
Sony           721
Zte            704
Blackshark     538
Name: count, dtype: int64


In [15]:
df['platform'].unique()

<ArrowStringArray>
['Tokopedia', 'Shopee', 'Blibli', 'Bliblii', 'Baru']
Length: 5, dtype: str

In [16]:
df['platform'] = df['platform'].replace({'Bliblii': 'Blibli', 'Baru': np.nan})
df['platform'] = df['platform'].str.strip().str.title()

print("Platform setelah fix:")
print(df['platform'].value_counts(dropna=False))

Platform setelah fix:
platform
Blibli       12218
Tokopedia    11705
Shopee       11650
NaN             78
Name: count, dtype: int64


In [17]:
print("Distribusi kondisi per platform (sebelum fillna):")
print(df.groupby(['platform', 'kondisi'], dropna=False).size().unstack(fill_value=0))

df['kondisi'] = df['kondisi'].fillna('Baru')

Distribusi kondisi per platform (sebelum fillna):
kondisi    Baru  Bekas   NaN
platform                    
Blibli     9599      0  2619
Shopee     6382   5268     0
Tokopedia  5950   5755     0
NaN           0      0    78


In [18]:
df.isnull().sum()

nama_produk           0
harga                 0
rating            18708
jumlah_terjual        0
lokasi_penjual    10128
brand                 0
kategori              0
kondisi               0
platform             78
dtype: int64

In [19]:
df['lokasi_penjual'] = df['lokasi_penjual'].fillna('Tidak Diketahui')
df['lokasi_penjual'] = df['lokasi_penjual'].str.strip()

In [20]:
df['brand']    = df['brand'].str.strip().str.title()
df['kategori'] = df['kategori'].str.strip().str.title()
df['kondisi']  = df['kondisi'].str.strip().str.title()

print(f"Shape final setelah semua cleaning: {df.shape}")
df.isnull().sum()

Shape final setelah semua cleaning: (35651, 9)


nama_produk           0
harga                 0
rating            18708
jumlah_terjual        0
lokasi_penjual        0
brand                 0
kategori              0
kondisi               0
platform             78
dtype: int64

In [21]:
df[df['platform'].isna()][['nama_produk', 'brand', 'kategori', 'kondisi', 'harga', 'platform']].head(20)

,nama_produk,brand,kategori,kondisi,harga,platform
82597,Hp Asus Zenfone 11 Ultra Ram 12GB Internal 256...,Asus,Hp,Baru,9999000,NaN
82598,Hp Asus Zenfone 11 Ultra Ram 16GB Internal 512...,Asus,Hp,Baru,13999000,NaN
82599,Asus ROG Phone 7 Smartphone [8GB/256GB],Asus,Hp,Baru,10499000,NaN
82600,Asus ZenFone Go ZB450KL Smartphone - Sheer Gold,Asus,Hp,Baru,1200000,NaN
82601,Asus ZenFone 4 Selfie ZD553KL Smartphone - Ros...,Asus,Hp,Baru,3499000,NaN
82602,Asus Zenfone Go ZB450KL Smartphone - Silver [1...,Asus,Hp,Baru,1200000,NaN
82603,Asus ROG II Smartphone,Asus,Hp,Baru,9000000,NaN
82604,Asus Zenfone 2 Laser ZE500KG Putih Smartphone ...,Asus,Hp,Baru,1835000,NaN
82605,Asus Zenfone Live L1 Smartphone [16GB/2GB] Gold,Asus,Hp,Baru,1600000,NaN
82606,Asus ZenFone 5 ZE620KL Smartphone [4GB/ 64GB],Asus,Hp,Baru,4750000,NaN


In [22]:
placeholder_mask = df['nama_produk'].str.strip().str.lower().isin(
    ['pre-order', 'pre order', 'preorder']
)
print(f"Drop pre-order placeholder: {placeholder_mask.sum()} baris")
df = df[~placeholder_mask]

# Flag pre-order dengan detail lengkap
df['is_preorder'] = df['nama_produk'].str.contains(
    r'pre[- ]?order|p/o|\bpo\b|\[po\]', case=False, na=False, regex=True
)
print(f"Pre-order diflag (tetap di data): {df['is_preorder'].sum()} baris")

# DataFrame bersih untuk analisis harga & revenue
df_analisis = df[~df['is_preorder']].copy()
print(f"Shape df_analisis (tanpa pre-order): {df_analisis.shape}")

Drop pre-order placeholder: 90 baris
Pre-order diflag (tetap di data): 13 baris
Shape df_analisis (tanpa pre-order): (35548, 10)


In [23]:
def hitung_skor_fuzzy(row):
    nama_produk = str(row['nama_produk']).lower()
    brand       = str(row['brand']).lower()
    return fuzz.token_set_ratio(brand, nama_produk)

df_analisis = df_analisis.copy()
df_analisis['skor_kemiripan'] = df_analisis.apply(hitung_skor_fuzzy, axis=1)

brand_tidak_sesuai = df_analisis[df_analisis['skor_kemiripan'] < 70]
print(f"Produk dicurigai salah brand: {len(brand_tidak_sesuai)}")
display(brand_tidak_sesuai[['nama_produk', 'brand', 'skor_kemiripan']].head(20))

Produk dicurigai salah brand: 6041


,nama_produk,brand,skor_kemiripan
0,motorolla MOTO G45 5G RAM 8+8/256GB Garansi Re...,Zte,7.142857
179,Black Shark GS3 Ultra Smartwatch | Dual-Band G...,Blackshark,13.422819
180,[NEW PRODUCT] Handphone Gaming REDMAGIC 11 PRO...,Blackshark,11.666667
181,HP Y73s 5G RAM 8GB+256GB/Cuci Gudang Handphon...,Blackshark,12.698413
182,itel City 200 NFC IP65 | 6.78” 120Hz Display |...,Blackshark,13.913043
183,Zte Blade Nubia A36 Ram 4+4/64 Garansi Resmi N...,Blackshark,20.512821
184,ITEL SUPER 26 ULTRA - T7300 Ultra Gaming Proce...,Blackshark,10.256410
185,(DCG) ZTE NUBIA NEO 3 GT 5G 8/256 GB GARANSI ...,Blackshark,12.345679
186,nubia V70 NFC RAM 8GB+12GB ROM 256GB 108MP Ult...,Blackshark,8.805031
187,"TECNO POVA 7 5G - 8+8GB*/256GB, 6000 mAh, Medi...",Blackshark,10.285714


Daftar brand yang dikenal di dataset

In [24]:
known_brands = [
    'Apple', 'Samsung', 'Xiaomi', 'Oppo', 'Vivo', 'Realme', 'Nokia',
    'Infinix', 'Tecno', 'Itel', 'Poco', 'Honor', 'Huawei', 'Blackshark',
    'Asus', 'Acer', 'Lenovo', 'Dell', 'Hp', 'Msi', 'Microsoft', 'Axioo',
    'Advan', 'Zyrex', 'Sony', 'Zte'
]

Khusus iPhone/Macbook/iPad → Apple

In [25]:
brand_alias = {
    'Iphone': 'Apple',
    'Macbook': 'Apple',
    'Ipad': 'Apple',
}

In [26]:
def ekstrak_brand_dari_nama(nama_produk, brand_existing):
    if pd.isna(nama_produk):
        return brand_existing

    nama_lower = nama_produk.lower()

    for alias, brand_fix in brand_alias.items():
        if alias.lower() in nama_lower:
            return brand_fix

    for brand in known_brands:
        if brand.lower() in nama_lower:
            return brand

    return brand_existing

Hanya terapkan ke baris yang mismatch (skor_kemiripan < 70)

In [27]:
mismatch_mask = df_analisis['skor_kemiripan'] < 70

df_analisis.loc[mismatch_mask, 'brand'] = df_analisis[mismatch_mask].apply(
    lambda r: ekstrak_brand_dari_nama(r['nama_produk'], r['brand']), axis=1
)

print(f"Baris yang dicoba difix: {mismatch_mask.sum()}")

# Hitung berapa yang berhasil diidentifikasi brandnya dari nama produk
berhasil = df_analisis[mismatch_mask]['brand'].isin(known_brands)
print(f"Berhasil difix: {berhasil.sum()}")
print(f"Tidak ketemu brand di nama produk: {(~berhasil).sum()} baris")

Baris yang dicoba difix: 6041
Berhasil difix: 6041
Tidak ketemu brand di nama produk: 0 baris


#### **CEK HASIL FIX**

Apakah brand baru benar2 cocok sama nama produk?

In [28]:
hasil_fix = df_analisis[mismatch_mask].copy()

print(f"Total yang dicoba fix: {len(hasil_fix)}")
print(f"Berhasil ketemu brand di nama produk: {berhasil.sum()}")
print(f"Tidak ketemu (brand lama dipertahankan): {(~berhasil).sum()}")

print("\n--- Sample yang BERHASIL difix ---")
display(
    hasil_fix[berhasil][['nama_produk', 'brand']].head(20)
)

print("\n--- Sample yang TIDAK ketemu brand (brand lama dipertahankan) ---")
display(
    hasil_fix[~berhasil][['nama_produk', 'brand']].head(20)
)

Total yang dicoba fix: 6041
Berhasil ketemu brand di nama produk: 6041
Tidak ketemu (brand lama dipertahankan): 0

--- Sample yang BERHASIL difix ---


,nama_produk,brand
0,motorolla MOTO G45 5G RAM 8+8/256GB Garansi Re...,Zte
179,Black Shark GS3 Ultra Smartwatch | Dual-Band G...,Blackshark
180,[NEW PRODUCT] Handphone Gaming REDMAGIC 11 PRO...,Blackshark
181,HP Y73s 5G RAM 8GB+256GB/Cuci Gudang Handphon...,Hp
182,itel City 200 NFC IP65 | 6.78” 120Hz Display |...,Itel
183,Zte Blade Nubia A36 Ram 4+4/64 Garansi Resmi N...,Zte
184,ITEL SUPER 26 ULTRA - T7300 Ultra Gaming Proce...,Itel
185,(DCG) ZTE NUBIA NEO 3 GT 5G 8/256 GB GARANSI ...,Zte
186,nubia V70 NFC RAM 8GB+12GB ROM 256GB 108MP Ult...,Blackshark
187,"TECNO POVA 7 5G - 8+8GB*/256GB, 6000 mAh, Medi...",Tecno



--- Sample yang TIDAK ketemu brand (brand lama dipertahankan) ---


,nama_produk,brand


#### **DETEKSI PRODUK SMARTWATCH YANG TERSESAT DI KATEGORI HP**

Pola "watch jadi objek utama" vs "watch sebagai bonus/gratis

In [29]:
pola_bonus = r'gratis|bonus|free gift|free\s+smartwatch'

mask_watch_utama = (
    df_analisis['nama_produk'].str.contains(r'\bwatch\b|smartwatch', case=False, na=False, regex=True)
    & ~df_analisis['nama_produk'].str.contains(pola_bonus, case=False, na=False, regex=True)
)

print(f"Produk smartwatch yang salah kategori: {mask_watch_utama.sum()}")
display(df_analisis[mask_watch_utama][['nama_produk', 'brand', 'kategori']])

Produk smartwatch yang salah kategori: 20


,nama_produk,brand,kategori
179,Black Shark GS3 Ultra Smartwatch | Dual-Band G...,Blackshark,Hp
193,"Black Shark Watch S3 | 1.43"" AMOLED Smartwatch...",Blackshark,Hp
34395,REDMI WATCH 5 ACTIVE GARANSI RESMI XIAOMI,Xiaomi,Hp
34412,REDMI WATCH 5LITE GARANSI RESMI XIAOMI,Xiaomi,Hp
54618,HUAWEI WATCH BUDS GARANSI RESMI,Huawei,Hp
54625,HUAWEI WATCH FIT 2 ACTIVE | GARANSI RESMI,Huawei,Hp
54628,HUAWEI WATCH GT 3 PRO | GARANSI RESMI,Huawei,Hp
54636,HUAWEI WATCH FIT 3 [GREEN] - GARANSI RESMI,Huawei,Hp
54637,HUAWEI WATCH FIT 3 [WHITE] - GARANSI RESMI,Huawei,Hp
54638,HUAWEI WATCH FIT 3 [PINK] - GARANSI RESMI,Huawei,Hp


#### **KEPUTUSAN**

Drop dari analisis HP & Laptop, karena kategori produk yag ada cuma 2: 'Hp' dan 'Laptop', smartwatch bukan keduanya

In [30]:
print(f"Shape sebelum drop smartwatch: {df_analisis.shape}")
df_analisis = df_analisis[~mask_watch_utama]
print(f"Shape setelah drop smartwatch: {df_analisis.shape}")

Shape sebelum drop smartwatch: (35548, 11)
Shape setelah drop smartwatch: (35528, 11)


Cek brand-brand yang sering muncul di nama produk tapi tidak ada di known_brands

In [31]:
def cek_brand_tidak_dikenal(nama_produk):
    nama_lower = str(nama_produk).lower()
    pola_brand_umum = r'\b(doogee|nubia|black\s?shark|redmagic|infinix|cubot|ulefone|blackview|oukitel|tecno)\b'
    match = re.findall(pola_brand_umum, nama_lower)
    return match

df_analisis['brand_potensial'] = df_analisis['nama_produk'].apply(cek_brand_tidak_dikenal)
brand_baru_ditemukan = df_analisis[df_analisis['brand_potensial'].apply(len) > 0]

print("Brand yang mungkin belum tertangkap known_brands:")
print(brand_baru_ditemukan['brand_potensial'].explode().value_counts())

Brand yang mungkin belum tertangkap known_brands:
brand_potensial
infinix        1831
tecno          1003
nubia           519
black shark     163
blackshark      106
redmagic         10
blackview         2
doogee            1
Name: count, dtype: int64


#### **UPDATE known_brands & brand_alias**

In [32]:
known_brands = [
    'Apple', 'Samsung', 'Xiaomi', 'Oppo', 'Vivo', 'Realme', 'Nokia',
    'Infinix', 'Tecno', 'Itel', 'Poco', 'Honor', 'Huawei', 'Blackshark',
    'Asus', 'Acer', 'Lenovo', 'Dell', 'Hp', 'Msi', 'Microsoft', 'Axioo',
    'Advan', 'Zyrex', 'Sony', 'Zte',
    
    # Brand baru yang ditemukan dari nama produk:
    'Blackview', 'Doogee'
    # NOTE: Nubia & Redmagic TIDAK dimasukkan sebagai known_brands tersendiri
    # karena akan di-mapping ke ZTE via brand_alias di bawah
]

brand_alias = {
    'Iphone'      : 'Apple',
    'Macbook'     : 'Apple',
    'Ipad'        : 'Apple',
    'Black Shark' : 'Blackshark',   # normalisasi spasi
    'Nubia'       : 'Zte',          # sub-brand ZTE
    'Redmagic'    : 'Zte',          # sub-brand ZTE (Red Magic)
    'Red Magic'   : 'Zte',          # variasi penulisan dengan spasi
}

def ekstrak_brand_dari_nama(nama_produk, brand_existing):
    if pd.isna(nama_produk):
        return brand_existing

    nama_lower = nama_produk.lower()

    # Cek alias dulu (termasuk Nubia/Redmagic → Zte)
    for alias, brand_fix in brand_alias.items():
        if alias.lower() in nama_lower:
            return brand_fix

    # Cek known brands
    for brand in known_brands:
        if brand.lower() in nama_lower:
            return brand

    return brand_existing

RE-RUN fix untuk baris mismatch dengan known_brands terbaru

In [33]:
df_analisis.loc[mismatch_mask, 'brand'] = df_analisis[mismatch_mask].apply(
    lambda r: ekstrak_brand_dari_nama(r['nama_produk'], r['brand']), axis=1
)

print(f"Baris yang dicoba difix: {mismatch_mask.sum()}")
berhasil = df_analisis[mismatch_mask]['brand'].isin(known_brands + ['Zte'])
print(f"Berhasil difix: {berhasil.sum()}")
print(f"Masih belum ketemu brand: {(~berhasil).sum()}")

C:\Users\Khairunisa Olive\AppData\Local\Temp\ipykernel_29472\3801791705.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_analisis.loc[mismatch_mask, 'brand'] = df_analisis[mismatch_mask].apply(
C:\Users\Khairunisa Olive\AppData\Local\Temp\ipykernel_29472\3801791705.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  berhasil = df_analisis[mismatch_mask]['brand'].isin(known_brands + ['Zte'])


Baris yang dicoba difix: 6041
Berhasil difix: 6037
Masih belum ketemu brand: 0


#### **VERIFIKASI**

Cek ulang sample yang Nubia/Redmagic

In [34]:
mask_nubia_redmagic = df_analisis['nama_produk'].str.contains(
    r'nubia|redmagic|red magic', case=False, na=False, regex=True
)
print("Brand untuk produk Nubia/Redmagic setelah fix:")
print(df_analisis[mask_nubia_redmagic]['brand'].value_counts())

display(df_analisis[mask_nubia_redmagic][['nama_produk', 'brand']].head(10))

Brand untuk produk Nubia/Redmagic setelah fix:
brand
Zte           463
Asus            1
Blackshark      1
Name: count, dtype: int64


,nama_produk,brand
1,ZTE Nubia Z50 (5G) (12GB/256GB) (The Best Of C...,Zte
2,ZTE Nubia Redmagic 8s Pro 16GB/512GB Garansi R...,Zte
3,ZTE Nubia Z50 (5G) (12GB/256GB) (Jaminan Produ...,Zte
4,Digitalisme x Promo Nubia RedMagic 11 Pro 16GB...,Zte
6,ZTE NUBIA Neo 3 GT 5G (8/256GB) 6000mAH Super ...,Zte
7,ZTE NUBIA NEO 3 GT [5G] RAM [8/256GB] - ZTE Nu...,Zte
8,(Exclusive Devgadget) ZTE NUBIA A36 (4+8)GB/64...,Zte
10,(Exclusive Creators) ZTE NUBIA V70 DESIGN 8/25...,Zte
11,(Exclusive Devgadget) ZTE NUBIA V80 Max 6GB/12...,Zte
12,ZTE Nubia Redmagic 8S Pro 16GB/512GB Garansi R...,Zte


#### **DROP PRODUK BLACKVIEW & DOOGEE**

Brand minor yang tidak ada di known_brands, datanya sangat sedikit (6 baris)

In [35]:
mask_drop_minor_brand = df_analisis['nama_produk'].str.contains(
    r'blackview|doogee', case=False, na=False, regex=True
)

print(f"Shape sebelum drop Blackview/Doogee: {df_analisis.shape}")
print(f"Baris yang di-drop: {mask_drop_minor_brand.sum()}")

df_analisis = df_analisis[~mask_drop_minor_brand]
print(f"Shape setelah drop: {df_analisis.shape}")

Shape sebelum drop Blackview/Doogee: (35528, 12)
Baris yang di-drop: 3
Shape setelah drop: (35525, 12)


In [36]:
known_brands = [
    'Apple', 'Samsung', 'Xiaomi', 'Oppo', 'Vivo', 'Realme', 'Nokia',
    'Infinix', 'Tecno', 'Itel', 'Poco', 'Honor', 'Huawei', 'Blackshark',
    'Asus', 'Acer', 'Lenovo', 'Dell', 'Hp', 'Msi', 'Microsoft', 'Axioo',
    'Advan', 'Zyrex', 'Sony', 'Zte'
]

In [37]:
df_analisis

,nama_produk,harga,rating,jumlah_terjual,lokasi_penjual,brand,kategori,kondisi,platform,is_preorder,skor_kemiripan,brand_potensial
0,motorolla MOTO G45 5G RAM 8+8/256GB Garansi Re...,1500000,NaN,250,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia,False,7.142857,[]
1,ZTE Nubia Z50 (5G) (12GB/256GB) (The Best Of C...,9499000,NaN,0,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia,False,100.000000,[nubia]
2,ZTE Nubia Redmagic 8s Pro 16GB/512GB Garansi R...,14999000,NaN,0,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia,False,100.000000,"[nubia, redmagic]"
3,ZTE Nubia Z50 (5G) (12GB/256GB) (Jaminan Produ...,9499000,NaN,0,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia,False,100.000000,[nubia]
4,Digitalisme x Promo Nubia RedMagic 11 Pro 16GB...,17652000,5.0,6,Kota Administrasi Jakarta Pusat,Zte,Hp,Baru,Tokopedia,False,100.000000,"[nubia, redmagic, nubia]"
...,...,...,...,...,...,...,...,...,...,...,...,...
94347,ZTE F670L Port Hijau Ijo Unit Only,160000,NaN,0,Tidak Diketahui,Zte,Hp,Bekas,Shopee,False,100.000000,[]
94349,ZTE F670L,165000,5.0,139,Tidak Diketahui,Zte,Hp,Bekas,Shopee,False,100.000000,[]
94350,ZTE Nubia exAndromax,121000,5.0,1,Tangerang,Zte,Hp,Bekas,Shopee,False,100.000000,[nubia]
94351,ZTE MYREPUBIK ZTE F670L,160000,NaN,2,Tidak Diketahui,Zte,Hp,Bekas,Shopee,False,100.000000,[]


In [38]:
df_analisis = df_analisis.drop(columns=['skor_kemiripan', 'brand_potensial'])

print("Kolom setelah dibersihkan:")
print(df_analisis.columns.tolist())

Kolom setelah dibersihkan:
['nama_produk', 'harga', 'rating', 'jumlah_terjual', 'lokasi_penjual', 'brand', 'kategori', 'kondisi', 'platform', 'is_preorder']


In [39]:
mask_motorola = df_analisis['nama_produk'].str.contains(
    r'motorol|moto g\d|moto e\d|moto x\d|moto razr|thinkphone',
    case=False, na=False, regex=True
)

print(f"Shape sebelum drop Motorola: {df_analisis.shape}")
print(f"Baris yang di-drop: {mask_motorola.sum()}")

df_analisis = df_analisis[~mask_motorola]
print(f"Shape setelah drop: {df_analisis.shape}")

Shape sebelum drop Motorola: (35525, 10)
Baris yang di-drop: 67
Shape setelah drop: (35458, 10)


In [40]:
df_analisis

,nama_produk,harga,rating,jumlah_terjual,lokasi_penjual,brand,kategori,kondisi,platform,is_preorder
1,ZTE Nubia Z50 (5G) (12GB/256GB) (The Best Of C...,9499000,NaN,0,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia,False
2,ZTE Nubia Redmagic 8s Pro 16GB/512GB Garansi R...,14999000,NaN,0,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia,False
3,ZTE Nubia Z50 (5G) (12GB/256GB) (Jaminan Produ...,9499000,NaN,0,Kota Administrasi Jakarta Barat,Zte,Hp,Baru,Tokopedia,False
4,Digitalisme x Promo Nubia RedMagic 11 Pro 16GB...,17652000,5.0,6,Kota Administrasi Jakarta Pusat,Zte,Hp,Baru,Tokopedia,False
5,ZTE AXON 30 ULTRA 5G SD888 12GB 256GB 128GB BL...,70652400,NaN,0,Kota Administrasi Jakarta Selatan,Zte,Hp,Baru,Tokopedia,False
...,...,...,...,...,...,...,...,...,...,...
94347,ZTE F670L Port Hijau Ijo Unit Only,160000,NaN,0,Tidak Diketahui,Zte,Hp,Bekas,Shopee,False
94349,ZTE F670L,165000,5.0,139,Tidak Diketahui,Zte,Hp,Bekas,Shopee,False
94350,ZTE Nubia exAndromax,121000,5.0,1,Tangerang,Zte,Hp,Bekas,Shopee,False
94351,ZTE MYREPUBIK ZTE F670L,160000,NaN,2,Tidak Diketahui,Zte,Hp,Bekas,Shopee,False


In [41]:
df_analisis = df_analisis.drop(columns=['is_preorder'])

In [42]:
df_analisis.to_csv('../data/ecommerce_product_dataset.csv', index=False)